# OutbreakSignal DE — Exploratory Data Analysis (EDA)
## OpenDengue Dataset (National Extract V1.3)

**Project:** OutbreakSignal DE — Microwave Group (AIO 2026, Module 4)
**Target:** Ingestion and analysis of epidemiological ground-truth for Dengue in Southeast Asia.

### Notebook Objectives
1. **Schema & Data Quality Audit**: Assess columns, data types, missingness, and subtle traps (e.g. naming conventions, case definitions).
2. **Temporal Resolution Resolution (T_res)**: Analyze mixing of , , and  records and harmonize time-series.
3. **Southeast Asia (SEA) Epidemiological Trends**: Long-term surveillance trajectories (1955–2025) and outbreak detection across 11 nations.
4. **Vietnam In-Depth Case Study**: Historical peaks (1987, 2017, 2019, 2022) and monthly seasonality profiles.
5. **Silver Layer Recommendations**: Concrete design requirements for the upcoming PySpark standardization pipeline.

In [ ]:
import os
import io
import zipfile
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ZIP_PATH = BASE_DIR / "output" / "bronze_test" / "opendengue_national_raw.zip"
EDA_OUT = BASE_DIR / "output" / "eda"
EDA_OUT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {BASE_DIR}")
print(f"Raw zip path: {ZIP_PATH}")

## 1. Data Ingestion & Schema Inspection

In [ ]:
# Load CSV directly from the cached zip file
with zipfile.ZipFile(ZIP_PATH) as zf:
    csv_name = [n for n in zf.namelist() if n.lower().endswith(".csv")][0]
    with zf.open(csv_name) as f:
        df = pd.read_csv(f)

print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head(5))
df.info()

## 2. Data Quality & Critical Pipeline Traps

> ⚠️ **CRITICAL FINDING 1: The Whitespace Naming Trap**
> In OpenDengue, Vietnam is recorded as  (with a space), NOT .
> Simple exact string equality  drops all 317 records for Vietnam!

> ⚠️ **CRITICAL FINDING 2: Mixed Temporal Resolutions ()**
> The column  contains weekly, monthly, and annual counts mixed together.
> Summing raw records directly without grouping by  or aggregating by Year creates invalid comparisons.

> ⚠️ **CRITICAL FINDING 3: 100% Missing Sub-national Fields**
> , , and  have 29,873 / 29,873 null values (100% NaN) because this is the national level extract ().

In [ ]:
# Verify Missingness
missing = df.isna().sum()
print("--- Columns with Missing Values ---")
print(missing[missing > 0])

# Temporal Resolution Breakdown
print("
--- Temporal Resolutions (T_res) ---")
print(df["T_res"].value_counts())

# Case Definitions (Note casing inconsistency: confirmed vs Confirmed)
print("
--- Case Definitions Standardised ---")
print(df["case_definition_standardised"].value_counts())

# Verify Zero date duplicates
dups = df.duplicated(subset=["adm_0_name", "calendar_start_date", "calendar_end_date"], keep=False).sum()
print(f"
Duplicate (Country + StartDate + EndDate) Records: {dups}")

## 3. Southeast Asia (SEA) Regional Analysis

We filter the 11 countries of Southeast Asia using both standard ISO names and the OpenDengue spelling.

In [ ]:
SEA_COUNTRIES = [
    "BRUNEI DARUSSALAM", "CAMBODIA", "INDONESIA", "LAO PEOPLE'S DEMOCRATIC REPUBLIC",
    "MALAYSIA", "MYANMAR", "PHILIPPINES", "SINGAPORE", "THAILAND", "TIMOR-LESTE", "VIET NAM"
]

df_sea = df[df["adm_0_name"].isin(SEA_COUNTRIES)].copy()
df_sea["calendar_start_date"] = pd.to_datetime(df_sea["calendar_start_date"])
df_sea["calendar_end_date"] = pd.to_datetime(df_sea["calendar_end_date"])
df_sea["Year"] = df_sea["Year"].astype(int)

sea_summary = df_sea.groupby("adm_0_name").agg(
    records=("dengue_total", "count"),
    first_year=("Year", "min"),
    last_year=("Year", "max"),
    total_cases=("dengue_total", "sum"),
    weekly_records=("T_res", lambda s: (s == "Week").sum()),
    monthly_records=("T_res", lambda s: (s == "Month").sum()),
    yearly_records=("T_res", lambda s: (s == "Year").sum())
).sort_values(by="total_cases", ascending=False)

print(f"Total SEA Records: {len(df_sea):,}")
display(sea_summary)

### Visualizing Surveillance Coverage & Temporal Resolution

In [ ]:
plt.figure(figsize=(12, 6))
palette = {"Week": "#e74c3c", "Month": "#f39c12", "Year": "#3498db"}
country_order = df_sea.groupby("adm_0_name")["Year"].min().sort_values().index.tolist()

for country in country_order:
    c_df = df_sea[df_sea["adm_0_name"] == country]
    for res, grp in c_df.groupby("T_res"):
        plt.scatter(grp["Year"], [country]*len(grp), color=palette.get(res, "#95a5a6"), label=res, s=20, alpha=0.7)

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), title="Resolution (T_res)", loc="upper left", frameon=True)
plt.title("OpenDengue V1.3: Historical Surveillance Timeline per SEA Country", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Year")
plt.tight_layout()
plt.show()

## 4. Harmonized Annual Dengue Outbreak Trajectories

Because OpenDengue uses non-overlapping temporal resolutions within each country and calendar year, summing  by  reliably produces normalized annual caseloads.

In [ ]:
annual_sea = df_sea.groupby(["adm_0_name", "Year"]).agg(
    annual_cases=("dengue_total", "sum"),
    records=("dengue_total", "count")
).reset_index()

plt.figure(figsize=(14, 7))
major_sea = ["VIET NAM", "PHILIPPINES", "THAILAND", "INDONESIA", "MALAYSIA", "SINGAPORE"]
colors = sns.color_palette("tab10", len(major_sea))

for country, color in zip(major_sea, colors):
    sub = annual_sea[(annual_sea["adm_0_name"] == country) & (annual_sea["Year"] >= 1980)].sort_values("Year")
    plt.plot(sub["Year"], sub["annual_cases"], marker="o", markersize=4, label=country, color=color, linewidth=2)

plt.yscale("log")
plt.title("Harmonized Annual Dengue Cases (Log Scale, 1980 - 2025)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Year")
plt.ylabel("Annual Cases (Log Scale)")
plt.axvspan(2018.5, 2019.5, color="#e74c3c", alpha=0.15, label="2019 Regional Pandemic Wave")
plt.axvspan(2021.5, 2022.5, color="#f39c12", alpha=0.15, label="2022 Vietnam Epidemic")
plt.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

## 5. Vietnam In-Depth Outbreak & Seasonality Profile

In [ ]:
vn_annual = annual_sea[annual_sea["adm_0_name"] == "VIET NAM"].sort_values("Year")
vn_monthly = df_sea[(df_sea["adm_0_name"] == "VIET NAM") & (df_sea["T_res"] == "Month")].copy()
vn_monthly["month"] = vn_monthly["calendar_start_date"].dt.month

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot of annual cases
ax1.bar(vn_annual["Year"], vn_annual["annual_cases"], color="#2c3e50", alpha=0.85)
ax1.set_title("Vietnam: Historical Annual Dengue Cases (1960 - 2025)", fontweight="bold")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Cases")
peaks = [(1987, 354517, "1987 (354k)"), (2017, 172232, "2017 (172k)"), (2019, 320702, "2019 (320k)"), (2022, 367729, "2022 (367k)")]
for yr, val, txt in peaks:
    ax1.annotate(txt, xy=(yr, val), xytext=(yr-3, val+35000), arrowprops=dict(facecolor="#e74c3c", shrink=0.08, width=1.2, headwidth=6), fontsize=8.5, fontweight="bold", color="#c0392b")

# Monthly seasonality curve
monthly_stats = vn_monthly.groupby("month")["dengue_total"].agg(["mean", "std"])
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ax2.plot(range(1, 13), monthly_stats["mean"], marker="o", color="#e67e22", linewidth=2.5, label="Mean Cases")
ax2.fill_between(range(1, 13), np.maximum(0, monthly_stats["mean"] - monthly_stats["std"]), monthly_stats["mean"] + monthly_stats["std"], color="#e67e22", alpha=0.2, label="±1 Std Dev")
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_names)
ax2.axvspan(6.5, 10.5, color="#e74c3c", alpha=0.1, label="Peak Monsoon Season")
ax2.set_title("Vietnam: Monthly Transmission Seasonality", fontweight="bold")
ax2.set_xlabel("Month")
ax2.set_ylabel("Average Monthly Cases")
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Regional Seasonality Comparison (Heatmap)

We compare relative monthly transmission intensity across Southeast Asian nations.

In [ ]:
df_sea_month = df_sea[df_sea["T_res"] == "Month"].copy()
df_sea_month["month"] = df_sea_month["calendar_start_date"].dt.month
country_month = df_sea_month.groupby(["adm_0_name", "month"])["dengue_total"].mean().reset_index()
pivot = country_month.pivot(index="adm_0_name", columns="month", values="dengue_total")
valid_countries = pivot.dropna(thresh=6).index
pivot_norm = pivot.loc[valid_countries].div(pivot.loc[valid_countries].max(axis=1), axis=0)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_norm, cmap="YlOrRd", annot=True, fmt=".2f", xticklabels=month_names, linewidths=0.5, cbar_kws={"label": "Normalized Intensity (0-1)"})
plt.title("Southeast Asia Dengue Seasonality Heatmap (Relative Monthly Peak)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Month")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 7. Conclusions & Silver Layer Implementation Plan

### Key Takeaways for OutbreakSignal DE
1. **Bronze Layer Ingestion**: Ingest raw data with  (strings) and preserve , , .
2. **Country Normalization in Silver**:
   - Standardize  ->  (or canonical ISO alpha-3 ).
   - Ensure mapping handles all 11 SEA countries.
3. **Temporal Resolution Strategy**:
   - Keep raw records in Silver with explicit  (, , ).
   - Provide resampled / standardized view (e.g. monthly aggregated or annual totals) so downstream analytics don't compare apples to oranges.
4. **Case Definition Standardization**:
   - Lowercase and normalize:  vs  -> .
5. **Outbreak Threshold Baseline**:
   - In Vietnam, the normal baseline is 2,000–3,500 cases/month. Any early surge exceeding 5,000 cases before June represents an early outbreak anomaly to correlate with news RSS signals!